# 🤖 Orion Knowledge Assistant
## Construyendo un agente KA con Agent Bricks

---

> **DATABRICKS · MOSAIC AI · LABORATORIO**
>
> En este laboratorio construirás un agente conversacional sobre documentación técnica usando el enfoque **declarativo** de Agent Bricks — sin escribir código de retrieval, sin configurar embeddings manualmente, sin gestionar índices.

| ⏱ ~40 minutos | 🛠 Databricks Free Trial | ⚙ Serverless Compute v5 |
|:---:|:---:|:---:|

---

## 🤖 El escenario: Orion A1

La empresa **Orion Robotics** fabrica el robot humanoide **Orion A1** para uso industrial. Sus ingenieros de campo necesitan respuestas rápidas y precisas sobre:

- Procedimientos de mantenimiento y recalibración
- Verificación de checksums de firmware
- Cumplimiento de normas de seguridad (ISO 13849-1)
- Resolución de problemas en campo

Hoy, esa información está dispersa en manuales técnicos, guías de compliance y documentación interna almacenada en Databricks. **Nuestro objetivo**: construir el **Orion Knowledge Assistant (OKA)**, un agente que responda preguntas técnicas basándose exclusivamente en esa documentación, con citas a la fuente.

---

## 🎯 Objetivos del laboratorio

Al finalizar este lab serás capaz de:

| # | Objetivo |
|---|----------|
| 1 | Crear un agente Knowledge Assistant desde la UI de Agent Bricks |
| 2 | Configurar fuentes de conocimiento desde Unity Catalog Volumes |
| 3 | Probar el agente e interpretar sus respuestas y citas |
| 4 | Identificar respuestas fuera del contexto y entender cómo el agente las maneja |
| 5 | Mejorar la calidad del agente usando labeled data y el ciclo ALHF |

---

## ✅ Requisitos

Antes de comenzar, verifica lo siguiente:

- [ ] Tienes acceso a un workspace de **Databricks Free Trial** (o workspace con Agent Bricks Preview habilitado)
- [ ] El compute está configurado como **Serverless** con **environment version 5**
- [ ] Puedes navegar a la sección **Agents** en el menú lateral izquierdo

> 💡 **Nota:** Agent Bricks está actualmente en **Preview (Beta)**. Asegúrate de que tu workspace tenga esta feature habilitada. En el Free Trial viene activa por defecto.

---

## ⚙️ Setup del workspace

> 📌 **Paso previo requerido:** Antes de ejecutar este notebook, asegúrate de haber ejecutado el notebook **`Lab_AgentBricks_Setup_Volume`** para crear el Volume y copiar los documentos de Orion. Si ya lo hiciste, continúa aquí.

La siguiente celda verifica que tu entorno está listo para el laboratorio.

In [ ]:
import os, sys

print(f"✅ Python {sys.version.split()[0]}")
print()

# Verificar que el Volume existe y tiene documentos
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "orion_docs"
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print("📋 Verificando entorno...")
print()

# Verificar Volume
if os.path.exists(volume_path):
    files = [f for f in os.listdir(volume_path) if not f.startswith(".")]
    if files:
        print(f"✅ Volume encontrado: {volume_path}")
        print(f"✅ Documentos disponibles: {len(files)} archivo(s)")
        for f in sorted(files):
            print(f"   📄 {f}")
    else:
        print(f"⚠️  Volume existe pero está vacío.")
        print(f"   Ejecuta primero OKA_00_Setup_Volume para copiar los documentos.")
else:
    print(f"❌ Volume no encontrado: {volume_path}")
    print(f"   Ejecuta primero OKA_00_Setup_Volume antes de continuar.")

print()
print("🚀 ¡Listo para comenzar el laboratorio!")

---
# Parte A — Crear el agente Orion KA
> ⏱ **~15 minutos** · Todo desde la UI de Databricks, sin código

En esta parte usaremos la interfaz de Agent Bricks para crear el agente de forma declarativa. Vamos a aprender qué significa que sea "declarativo": tú describes **qué** quieres, Agent Bricks decide **cómo** construirlo.

### Paso A1 — Navegar a Agents

1. En el menú lateral izquierdo de Databricks, busca **Agents**
2. Haz clic en el botón **Create Agent**
3. En la pantalla de selección, haz clic en la tarjeta **Knowledge Assistant**

![Knowledge Assistant](https://github.com/yaliagap/01-agentbricks-knowledge-assistant/blob/main/img/ka_selection.png?raw=true)

> 💡 **¿Por qué Knowledge Assistant?**  
> Es el brick optimizado para RAG sobre documentación. Maneja automáticamente el parsing, chunking, embeddings y generación de citas. Ideal para el caso de Orion donde la fuente de verdad son documentos técnicos.

---

### Paso A2 — Configurar el agente

Completa el formulario con los siguientes datos. Puedes copiar los textos directamente desde aquí:

#### Nombre
```
Orion_KA_Agent
```

#### Descripción del agente
```
El Orion KA Agent (OKA) ayuda a ingenieros y técnicos a encontrar respuestas precisas en los manuales internos, guías de mantenimiento y documentos de seguridad de Orion. Entrega respuestas verificadas con referencias a la fuente, reduciendo el tiempo de búsqueda y garantizando información confiable y consistente.
```

#### Fuente de conocimiento
- **Knowledge source type**: `Files in a Volume`
- **Source**: Selecciona el Unity Catalog Volume que contiene los documentos de Orion (`workspace.default.orion_docs`)
- **Knowledge source name**: `Documentos Orion`
- **Content description**:
```
Contiene documentación técnica de Orion, notas de ingeniería, manuales de mantenimiento, guías de cumplimiento normativo y preguntas frecuentes.
```

#### Instrucciones del agente *(opcional pero recomendado)*
```
Eres el Orion Knowledge Assistant (OKA). Responde con un tono claro, profesional y factual, apropiado para ingenieros y personal técnico. Usa únicamente información verificada de los documentos internos de Orion, e incluye referencias a la fuente cuando estén disponibles. Si la respuesta no se encuentra en la documentación, indícalo claramente y sugiere secciones relacionadas o próximos pasos. No especules, no hagas suposiciones ni proporciones información fuera del contexto entregado.
```

Finalmente, haz clic en **Create Agent**.

> ⏳ **Nota:** La creación del agente puede tardar hasta 10 minutos. Durante ese tiempo Agent Bricks está procesando los documentos, generando embeddings y construyendo el índice vectorial. Continuaremos en la siguiente sección mientras esperamos.

---

### ⏳ Mientras el agente se crea (~10 minutos)...

Agent Bricks no está simplemente guardando una configuración. En este momento está:

| Paso interno | ¿Qué hace? |
|---|---|
| **Parsing** | Usa `ai_parse_document` para extraer texto, tablas e imágenes de cada PDF/DOCX |
| **Chunking** | Divide los documentos en fragmentos optimizados para retrieval |
| **Embedding** | Genera vectores para cada chunk usando un modelo de embeddings (ej. GTE) |
| **Vector Search** | Crea y popula un índice en Mosaic AI Vector Search |
| **Model Serving** | Despliega el endpoint del agente con el LLM y la lógica de retrieval |

> 🔑 **Punto clave:** Esto es lo que harías manualmente con código en el enfoque Code-First. Agent Bricks lo hace por ti, y si cambias los documentos fuente, el índice se actualiza automáticamente.

---

---
## 🔍 ¿Qué está pasando por detrás?

Mientras el agente se construye, entendamos los componentes que Agent Bricks está orquestando automáticamente. Un Knowledge Assistant **no es una caja negra** — es un sistema compuesto de arquitectura nativa de Databricks.

![Componentes Agent Bricks](https://github.com/yaliagap/01-agentbricks-knowledge-assistant/blob/main/img/ka_components.png?raw=true)

> 💡 Estos componentes trabajan en segundo plano. Tú no los gestionas manualmente — Agent Bricks los configura, conecta y mantiene por ti.

---

### D1 — Data Ingestion and Parsing

La base del Knowledge Assistant son los datos almacenados en **Unity Catalog Volumes**.

- **Fuente:** El usuario selecciona un Volume con archivos (PDF, DOCX, HTML)
- **Parsing:** El sistema usa `ai_parse_document`, una función de Mosaic AI que extrae texto, tablas e imágenes de documentos complejos — incluso elementos visuales como gráficos en un PDF se convierten en contexto que el LLM puede entender

---

### D2 — Mosaic AI Vector Search

Una vez parseados, los documentos se indexan para retrieval.

- **Embeddings gestionados:** Agent Bricks selecciona automáticamente un modelo de embeddings (ej. GTE) y aprovisiona un índice en Mosaic AI Vector Search
- **Sincronización automática:** El índice es completamente gestionado. Cuando se añaden nuevos archivos al Volume fuente, el índice se actualiza solo — sin re-indexación manual

---

### D3 — The Reasoning Engine and Model Serving

La lógica del agente se aloja en **Model Serving**.

- **Inferencia:** Cuando el usuario hace una pregunta, el sistema convierte la query a vectores, recupera los chunks más relevantes del Vector Search, y los pasa al LLM como contexto
- **Citas:** El Knowledge Assistant está diseñado para proveer citas. Mapea la respuesta al archivo fuente específico en el Volume, permitiendo verificar la precisión de cada respuesta

---

### D4 — The Quality Loop (Review App & Evaluation)

Este es el diferenciador de Agent Bricks respecto a un RAG construido a mano.

- **Review App:** Una UI integrada donde los SMEs pueden chatear con el agente y dar feedback (👍 / 👎 / edición directa)
- **LLM Judges:** Mosaic AI Agent Evaluation corre "jueces" automáticos sobre los traces de interacción, midiendo métricas como *faithfulness* (¿alucinó el modelo?) y *correctness*
- **Optimización:** Agent Bricks usa el feedback recolectado para proponer actualizaciones a las instrucciones del sistema o configuración, mejorando las métricas de desempeño de forma iterativa

---

---
# Parte B — Probar el agente
> ⏱ **~10 minutos** · Usando la interfaz de chat de Agent Bricks

Una vez que el agente muestre estado **Ready**, usa el panel de chat integrado en la interfaz de Agent Bricks para probarlo.

### Preguntas de prueba

Prueba estas preguntas en orden. Cada una evalúa un aspecto distinto del agente:

---

**Pregunta 1** — Visión general del producto:
```
¿Cuáles son las principales capacidades y casos de uso del robot Orion A1?
```
*¿Qué esperar?* El agente debe sintetizar información del overview del producto con citas al documento correspondiente.

---

**Pregunta 2** — Procedimiento técnico específico:
```
¿Qué pasos debo seguir para gestionar correctamente la batería del Orion y maximizar su vida útil?
```
*¿Qué esperar?* Una respuesta estructurada con pasos concretos del manual de Battery Management, con referencias a secciones específicas.

---

**Pregunta 3** — Razonamiento sobre firmware:
```
¿Cómo puedo verificar que el firmware del controlador de movimiento de Orion está correctamente instalado?
```
*¿Qué esperar?* El agente debe referenciar la guía de firmware v6 y dar pasos de verificación precisos.

---

**Pregunta 4** — Fuera del contexto ⚠️:
```
¿Qué significa la luz roja parpadeante en el panel de Orion?
```
*¿Qué esperar?* **Esta información no existe en la documentación de Orion.** Un buen agente debe reconocerlo claramente y sugerir pasos alternativos, en lugar de inventar una respuesta.

> 💡 Esta pregunta es clave: demuestra que el agente tiene **límites bien definidos** y no alucina cuando no tiene información. Eso es exactamente lo que queremos en un entorno de misión crítica.

---
---

### 🔍 Qué observar en cada respuesta

| Aspecto | ¿Qué buscar? |
|---------|-------------|
| **Citas** | ¿El agente referencia el documento y sección específica? |
| **Tono** | ¿Responde de forma profesional y técnica, como se indicó en las instrucciones? |
| **Límites** | En la pregunta 4, ¿admite que no tiene la información o intenta adivinar? |
| **Consistencia** | ¿Las respuestas 1,2 y 3 son coherentes con el contenido real de los documentos? |

<div style="background:#FFF3E0; border-left:4px solid #E67E22; padding:12px 16px; border-radius:4px; margin:12px 0;">
  <strong>🤔 Reflexión:</strong> ¿Qué pasó con la pregunta de la luz roja? ¿El agente respondió correctamente o intentó adivinar? Eso nos lleva a la Parte C.
</div>

---

---
# Parte C — Mejorar la calidad con ALHF
> ⏱ **~15 minutos** · Agent Learning from Human Feedback

En esta parte vamos a enseñarle al agente cómo responder mejor ante la pregunta de la luz roja. Esto es el ciclo **ALHF** en acción: feedback humano → mejora automática del sistema.

### C1 — Agregar labeled data (ejemplos con guidelines)

Este método es ideal cuando sabes exactamente qué respuesta quieres para una pregunta específica.

**Pasos:**

1. En la interfaz del agente, navega a la pestaña **Examples**
2. Haz clic en **Add**
3. Ingresa la pregunta y presiona **Add**:
   ```
   ¿Qué significa la luz roja parpadeante en el panel de Orion?
   ```
4. Haz clic en la pregunta para abrir sus detalles
5. En la sección **Guidelines**, agrega las siguientes instrucciones una por una:

   > Informar al usuario que no existe una luz roja parpadeante en el panel de Orion

   > Pedirle al usuario que reinicie Orion retirando y volviendo a insertar la batería

   > Solicitar que verifique el color exacto de la luz

   > Pedir que observe si la luz sigue parpadeando después del reinicio

6. Haz clic en **Save**

**Prueba el agente nuevamente** con la misma pregunta. ¿Cambió la respuesta?

También puedes variarlo preguntando:
```
¿Qué significa la luz azul parpadeante en el panel de Orion?
```
Observa cómo el agente adapta su respuesta usando las guidelines como guía de comportamiento, aunque la pregunta sea ligeramente diferente.

> 🔑 **¿Qué acaba de pasar?**  
> Las guidelines no son respuestas fijas hardcodeadas. Agent Bricks las usa para ajustar el comportamiento del LLM ante situaciones similares. El agente aprendió un **patrón de respuesta**, no una respuesta literal.

---

### C2 — Expert Review (Revisión por expertos)

Este método es ideal para mejorar la calidad de forma continua en producción, donde los SMEs revisan respuestas reales del agente.

El flujo de Expert Review permite:
- Revisar respuestas del agente a preguntas de evaluación
- Dar feedback en lenguaje natural sobre la calidad de cada respuesta
- Agregar guidelines y expectativas de comportamiento
- Evaluar respuestas en términos de exactitud, completitud y tono

**Pasos:**

1. En la interfaz del agente, navega a la pestaña **Review**
2. Selecciona una conversación de prueba para revisar
3. Usa los controles de feedback (👍 / 👎 / edición) para evaluar cada respuesta
4. Agrega comentarios en lenguaje natural explicando por qué la respuesta es buena o qué debería mejorar
5. Guarda el feedback — Agent Bricks lo incorpora al ciclo de optimización

Para el paso a paso completo consulta la documentación oficial:  
[Agent Bricks: Knowledge Assistant — Step 3: Improve quality](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant#step-3-improve-quality)

> 💬 **Pregunta para reflexionar:**  
> ¿Cómo se reflejan las guidelines en los traces del agente? Navega a la pestaña **Traces** y observa cómo el sistema incorpora el feedback en el contexto que recibe el LLM. ¿Ves las instrucciones del sistema actualizadas?

> ✅ **Mejor práctica:** La mejora de calidad es un proceso iterativo. En producción, planifica múltiples rondas de recolección de feedback y refinamiento para alcanzar el desempeño óptimo del agente.

---

---
# 🧹 Cleanup — Eliminar recursos

> ⚠️ **Importante:** El agente KA crea un **Vector Search endpoint** y un **Model Serving endpoint** en segundo plano. Estos endpoints generan costos aunque no estén en uso activo. Elimínalos al terminar el lab.

**Pasos para eliminar el agente:**

1. Navega a **Agents** en el menú lateral
2. Localiza **Orion_Knowledge_Assistant**
3. Haz clic en el agente para abrir sus detalles
4. Selecciona **Delete** en el menú de opciones (esquina superior derecha)
5. Confirma la eliminación

Al eliminar el agente se eliminan también el Vector Search endpoint y el Model Serving endpoint asociados.

---

---
# ✅ Resumen del laboratorio

Completaste el ciclo completo de un agente Knowledge Assistant con Agent Bricks:

<div style="display:grid; grid-template-columns: 1fr 1fr 1fr; gap:16px; margin:16px 0;">

<div style="background:#E3F2FD; border-top:4px solid #1E88E5; padding:16px; border-radius:8px;">
  <div style="font-weight:700; color:#1E88E5; margin-bottom:8px;">Parte A — Crear</div>
  <div style="font-size:13px; color:#334155;">Configuraste el agente de forma declarativa: nombre, descripción, fuente de datos e instrucciones de comportamiento.</div>
</div>

<div style="background:#E8F5E9; border-top:4px solid #27AE60; padding:16px; border-radius:8px;">
  <div style="font-weight:700; color:#27AE60; margin-bottom:8px;">Parte B — Probar</div>
  <div style="font-size:13px; color:#334155;">Evaluaste el agente con preguntas dentro y fuera del contexto. Observaste citas, tono y manejo de límites del conocimiento.</div>
</div>

<div style="background:#F3E5F5; border-top:4px solid #8E44AD; padding:16px; border-radius:8px;">
  <div style="font-weight:700; color:#8E44AD; margin-bottom:8px;">Parte C — Mejorar</div>
  <div style="font-size:13px; color:#334155;">Usaste labeled data y guidelines para enseñarle al agente cómo responder mejor. Viste el ciclo ALHF en acción.</div>
</div>

</div>

## Próximos pasos

Si quieres seguir explorando Agent Bricks, considera:

1. **Expandir las fuentes de conocimiento** — Agrega más documentos al Volume y observa cómo el índice se actualiza automáticamente
2. **Explorar el Multi-Agent Supervisor** — Combina OKA con un Genie Agent para responder tanto preguntas sobre documentos como consultas sobre datos estructurados
3. **Revisar los traces en MLflow** — Cada interacción queda trazada; puedes analizar latencia, retrieval scores y calidad de respuestas

## Recursos

| Recurso | Link |
|---------|------|
| Documentación Agent Bricks | [docs.databricks.com](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant) |
| Mosaic AI Agent Evaluation | [docs.databricks.com](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/index.html) |
| MLflow Tracing | [mlflow.org](https://mlflow.org/docs/latest/llms/tracing/index.html) |

---

<div style="text-align:center; color:#8BA3BE; font-size:12px; margin-top:24px;">
  © 2026 · Laboratorio preparado para clase demo · Knowledge Assistant con Agent Bricks
</div>